# Les Réseaux de Neurones Convolutifs (CNN)

## Objectif
Ce TP a pour objectif de résoudre un problème de **reconnaissance d'images** en utilisant les réseaux de neurones convolutifs (CNN).

## Problème
Nous allons entraîner **3 CNN différents** capables de reconnaître des **chiffres écrits à la main** (0 à 9), en utilisant le dataset **MNIST** fourni par Keras.

## Les 3 architectures ciblées

| Modèle | Convolutions | Couches spéciales | Taux d'erreur attendu |
|---|---|---|---|
| Small CNN | 2 conv (64→32 filtres, 3×3) | Aucune | ~1.39% |
| Medium CNN | 1 conv (32 filtres, 5×5) | MaxPooling + Dropout | ~1.03% |
| Large CNN | 2 conv (30→15 filtres, 5×5→3×3) | 2×MaxPooling + Dropout | ~0.82% |

### imports

In [46]:
# Import
import numpy as np
from keras.datasets import mnist
from keras.utils import to_categorical
from keras import backend as K


K.set_image_data_format('channels_first')

seed = 7
np.random.seed(seed)

from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten
from keras.layers import Conv2D, MaxPooling2D
from keras.models import model_from_json
import os

### Chargement et Préparation des Données (MNIST)

In [43]:
def get_data_mnist():
    # Data
    (X_train, y_train), (X_test, y_test) = mnist.load_data()

    # Reshape
    X_train = X_train.reshape(X_train.shape[0], 1, 28, 28).astype('float32')
    X_test  = X_test.reshape(X_test.shape[0], 1, 28, 28).astype('float32')

    # Encoding
    y_train = to_categorical(y_train)
    y_test  = to_categorical(y_test)
    
    num_classes = y_test.shape[1]

    return (X_train, y_train), (X_test, y_test), num_classes

In [44]:
# Test 
(X_train, y_train), (X_test, y_test), num_classes = get_data_mnist()

print("X_train shape :", X_train.shape)  
print("X_test shape  :", X_test.shape)   
print("Nombre de classes :", num_classes)

X_train shape : (60000, 1, 28, 28)
X_test shape  : (10000, 1, 28, 28)
Nombre de classes : 10


## Small CNN 

### Architecture du Small CNN
- **Conv2D** : 64 filtres, kernel 3×3, activation ReLU
- **Conv2D** : 32 filtres, kernel 3×3, activation ReLU
- **Flatten** : mise à plat du volume 3D en vecteur 1D
- **Dense** : 10 neurones + activation Softmax 

### Paramètres d'entraînement
| Paramètre | Valeur | Rôle |
|---|---|---|
| `loss` | categorical_crossentropy | Mesure l'écart entre prédiction et réalité |
| `optimizer` | adam | Met à jour les poids pour minimiser le loss |
| `epochs` | 10 | Nombre de passages complets sur les données |
| `batch_size` | 200 | Nombre d'images analysées avant mise à jour |

### Error rate function

In [ ]:
def print_model_error_rate(model, X_test, y_test):
    # Évaluation
    scores = model.evaluate(X_test, y_test, verbose=0)
    print("Model score      : %.2f%%" % (scores[1] * 100))
    print("Model error rate : %.2f%%" % (100 - scores[1] * 100))

### Construction et Compilation: 

In [ ]:
def small_model(num_classes):
    # Construction
    model = Sequential()
    
    model.add(Conv2D(64, (3, 3), input_shape=(1, 28, 28), activation='relu'))
    model.add(Conv2D(32, (3, 3), activation='relu'))
    model.add(Flatten())
    model.add(Dense(num_classes, activation='softmax'))
    
    # Compilation du modèle
    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )
    
    return model

### Entraînement

In [ ]:
model_small = small_model(num_classes)
model_small.summary()

model_small.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=200
)

c:\Program Files\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 64, 26, 26)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 24, 24)     │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │       184,330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 203,434 (794.66 KB)

 Trainable params: 203,434 (794.66 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 160s 532ms/step - accuracy: 0.8360 - loss: 2.7176 - val_accuracy: 0.9733 - val_loss: 0.0905
Epoch 2/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 165s 550ms/step - accuracy: 0.9797 - loss: 0.0651 - val_accuracy: 0.9744 - val_loss: 0.0819
Epoch 3/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 175s 583ms/step - accuracy: 0.9885 - loss: 0.0363 - val_accuracy: 0.9746 - val_loss: 0.0864
Epoch 4/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 178s 593ms/step - accuracy: 0.9909 - loss: 0.0277 - val_accuracy: 0.9751 - val_loss: 0.1023
Epoch 5/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 177s 590ms/step - accuracy: 0.9924 - loss: 0.0219 - val_accuracy: 0.9774 - val_loss: 0.0990
Epoch 6/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 179s 597ms/step - accuracy: 0.9951 - loss: 0.0139 - val_accuracy: 0.9754 - val_loss: 0.1321
Epoch 7/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 177s 591ms/step - accuracy: 0.9944 - loss: 0.0172 - val_accuracy: 0.9779 - val_loss: 0.1150
Epoch 8/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 169s 564ms/step - accuracy: 0.9960 -

### Evaluation

In [ ]:
print("\n=== Small CNN ===")
print_model_error_rate(model_small, X_test, y_test)


=== Small CNN ===
Model score      : 97.56%
Model error rate : 2.44%


## Normalisation des Données

### Raisons
Les pixels des images ont des valeurs entre **0 et 255** (uint8).
On les divise par **255** pour obtenir des valeurs entre **0 et 1**.

### Avantages de la normalisation
- Réduit l'écart entre les valeurs extrêmes
- Évite les **overflow** dans les calculs (des valeurs > 1 peuvent tendre vers l'infini)
- Accélère la **convergence** lors de l'entraînement
- Améliore la **précision** du modèle

### Normalized data

In [ ]:
def get_data_mnist_normalized():
    # données
    (X_train, y_train), (X_test, y_test) = mnist.load_data()

    # Reshape
    X_train = X_train.reshape(X_train.shape[0], 1, 28, 28).astype('float32')
    X_test  = X_test.reshape(X_test.shape[0], 1, 28, 28).astype('float32')

    # Normalisation
    X_train = X_train / 255.0
    X_test  = X_test  / 255.0

    # encoding
    y_train = to_categorical(y_train)
    y_test  = to_categorical(y_test)

    num_classes = y_test.shape[1]  

    return (X_train, y_train), (X_test, y_test), num_classes

# loading
(X_train, y_train), (X_test, y_test), num_classes = get_data_mnist_normalized()

print("Valeur min X_train :", X_train.min())  
print("Valeur max X_train :", X_train.max())  


Valeur min X_train : 0.0
Valeur max X_train : 1.0


### Entrainement

In [ ]:


print("\n=== Small CNN avec normalisation ===")
model_small_norm = small_model(num_classes)

model_small_norm.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=200
)




=== Small CNN avec normalisation ===
Epoch 1/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 172s 571ms/step - accuracy: 0.8577 - loss: 0.4825 - val_accuracy: 0.9793 - val_loss: 0.0652
Epoch 2/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 176s 587ms/step - accuracy: 0.9805 - loss: 0.0662 - val_accuracy: 0.9831 - val_loss: 0.0515
Epoch 3/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 179s 599ms/step - accuracy: 0.9859 - loss: 0.0457 - val_accuracy: 0.9839 - val_loss: 0.0509
Epoch 4/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 174s 579ms/step - accuracy: 0.9895 - loss: 0.0346 - val_accuracy: 0.9855 - val_loss: 0.0448
Epoch 5/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 180s 600ms/step - accuracy: 0.9926 - loss: 0.0241 - val_accuracy: 0.9854 - val_loss: 0.0480
Epoch 6/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 178s 595ms/step - accuracy: 0.9942 - loss: 0.0180 - val_accuracy: 0.9856 - val_loss: 0.0520
Epoch 7/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 182s 606ms/step - accuracy: 0.9958 - loss: 0.0148 - val_accuracy: 0.9857 - val_loss: 0.0527
Epoch 8/10
300/300 ━━━━━━━━━━━━━━━━━━━

### Evaluation

In [ ]:

print("\n=== Comparaison ===")
print("Sans normalisation :")
print_model_error_rate(model_small, X_test, y_test)
print("\nAvec normalisation :")
print_model_error_rate(model_small_norm, X_test, y_test)


=== Comparaison ===
Sans normalisation :
Model score      : 92.41%
Model error rate : 7.59%

Avec normalisation :
Model score      : 98.71%
Model error rate : 1.29%


## Medium CNN

### Architecture du Medium CNN
| Couche | Détails |
|---|---|
| **Conv2D** | 32 filtres, kernel 5×5, activation ReLU |
| **MaxPooling2D** | Taille 2×2 |
| **Dropout** | Taux 0.2 (20% des neurones ignorés) |
| **Flatten** | Mise à plat en vecteur 1D |
| **Dense** | 128 neurones, activation ReLU |
| **Dense** | 10 neurones, activation Softmax |

### Nouvelles couches introduites
-  **MaxPooling2D** : réduit la dimension spatiale en gardant uniquement
  la valeur maximale dans chaque fenêtre 2×2 → moins de paramètres, plus robuste
-  **Dropout** : désactive aléatoirement 20% des neurones à chaque étape
  d'entraînement → évite le **surapprentissage (overfitting)**

### Construction et Compilation: 

In [ ]:
def medium_model(num_classes):
    # Construction
    model = Sequential()

    model.add(Conv2D(32, (5, 5), input_shape=(1, 28, 28), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(0.2))
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))

    # Compilation
    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model

model_medium = medium_model(num_classes)
model_medium.summary()



### Entraînement

In [ ]:
model_medium.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=200
)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 32, 24, 24)     │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 12, 12)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 12, 12)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 592,074 (2.26 MB)

 Trainable params: 592,074 (2.26 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 144s 479ms/step - accuracy: 0.8531 - loss: 0.5103 - val_accuracy: 0.9758 - val_loss: 0.0799
Epoch 2/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 141s 470ms/step - accuracy: 0.9773 - loss: 0.0732 - val_accuracy: 0.9841 - val_loss: 0.0474
Epoch 3/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 152s 506ms/step - accuracy: 0.9852 - loss: 0.0497 - val_accuracy: 0.9866 - val_loss: 0.0403
Epoch 4/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 156s 519ms/step - accuracy: 0.9880 - loss: 0.0382 - val_accuracy: 0.9885 - val_loss: 0.0353
Epoch 5/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 154s 514ms/step - accuracy: 0.9895 - loss: 0.0317 - val_accuracy: 0.9885 - val_loss: 0.0356
Epoch 6/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 155s 518ms/step - accuracy: 0.9922 - loss: 0.0251 - val_accuracy: 0.9884 - val_loss: 0.0335
Epoch 7/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 152s 507ms/step - accuracy: 0.9933 - loss: 0.0217 - val_accuracy: 0.9892 - val_loss: 0.0318
Epoch 8/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 155s 516ms/step - accuracy: 0.9944 -

### Evaluation

In [ ]:
print("\n=== Medium CNN ===")
print_model_error_rate(model_medium, X_test, y_test)


=== Medium CNN ===
Model score      : 98.90%
Model error rate : 1.10%


## Large CNN

### Architecture du Large CNN
| Couche | Détails |
|---|---|
| **Conv2D** | 30 filtres, kernel 5×5, activation ReLU |
| **MaxPooling2D** | Taille 2×2 |
| **Conv2D** | 15 filtres, kernel 3×3, activation ReLU |
| **Dropout** | Taux 0.2 (20% des neurones ignorés) |
| **Flatten** | Mise à plat en vecteur 1D |
| **Dense** | 128 neurones, activation ReLU |
| **Dense** | 50 neurones, activation ReLU |
| **Dense** | 10 neurones, activation Softmax |

### Différences clés par rapport au Medium CNN
-  **2 couches de convolution** au lieu d'une → extraction de features plus complexes
-  **2 couches Dense supplémentaire** (128 + 50) → classification plus fine
-  **Kernel plus grand** (5×5) pour la 1ère conv → capture des patterns plus larges

### Construction et Compilation: 

In [ ]:
def large_model(num_classes):
    # Construction
    model = Sequential()

    model.add(Conv2D(30, (5, 5), input_shape=(1, 28, 28), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(15, (3, 3), activation='relu'))
    model.add(Dropout(0.2))
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))

    # Compilation
    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model



### Entraînement

In [ ]:
model_large = large_model(num_classes)
model_large.summary()

model_large.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=200
)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_5 (Conv2D)               │ (None, 30, 24, 24)     │           780 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 12, 12)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 15, 10, 10)     │         4,065 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 15, 10, 10)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 1500)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       192,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 50)             │         6,450 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 10)             │           510 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 203,933 (796.61 KB)

 Trainable params: 203,933 (796.61 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 136s 452ms/step - accuracy: 0.8047 - loss: 0.6573 - val_accuracy: 0.9767 - val_loss: 0.0717
Epoch 2/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 140s 465ms/step - accuracy: 0.9760 - loss: 0.0772 - val_accuracy: 0.9861 - val_loss: 0.0414
Epoch 3/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 146s 486ms/step - accuracy: 0.9839 - loss: 0.0518 - val_accuracy: 0.9889 - val_loss: 0.0345
Epoch 4/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 146s 485ms/step - accuracy: 0.9879 - loss: 0.0403 - val_accuracy: 0.9896 - val_loss: 0.0319
Epoch 5/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 144s 481ms/step - accuracy: 0.9897 - loss: 0.0318 - val_accuracy: 0.9902 - val_loss: 0.0284
Epoch 6/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 144s 479ms/step - accuracy: 0.9910 - loss: 0.0278 - val_accuracy: 0.9900 - val_loss: 0.0296
Epoch 7/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 144s 481ms/step - accuracy: 0.9928 - loss: 0.0217 - val_accuracy: 0.9899 - val_loss: 0.0278
Epoch 8/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 145s 483ms/step - accuracy: 0.9933 -

### Evaluation

In [ ]:
print("\n=== Large CNN ===")
print_model_error_rate(model_large, X_test, y_test)


=== Large CNN ===
Model score      : 99.09%
Model error rate : 0.91%


## Sauvegarde et Chargement

### Save function

In [ ]:
def save_keras_model(model, filename):
    # dossier
    models_dir = "models"
    os.makedirs(models_dir, exist_ok=True)

    # arch
    json_path = os.path.join(models_dir, filename + ".json")
    model_json = model.to_json()
    with open(json_path, "w") as json_file:
        json_file.write(model_json)
    
    # poids
    weights_path = os.path.join(models_dir, filename + ".weights.h5")
    model.save_weights(weights_path)
    print("Modèle sauvegardé : %s + %s" % (json_path, weights_path))


### Load function

In [ ]:
def load_keras_model(filename):
    # Chargement
    models_dir = "models"
    json_path = os.path.join(models_dir, filename + ".json")
    weights_path = os.path.join(models_dir, filename + ".weights.h5")

    # Arch
    with open(json_path, 'r') as json_file:
        loaded_model_json = json_file.read()
    loaded_model = model_from_json(loaded_model_json)
    
    # Poids
    loaded_model.load_weights(weights_path)
    print("Modèle chargé : %s + %s" % (json_path, weights_path))
    
    return loaded_model


### Saving

In [ ]:
save_keras_model(model_small_norm, "small_cnn")
save_keras_model(model_medium,     "medium_cnn")
save_keras_model(model_large,      "large_cnn")

Modèle sauvegardé : models\small_cnn.json + models\small_cnn.weights.h5
Modèle sauvegardé : models\medium_cnn.json + models\medium_cnn.weights.h5
Modèle sauvegardé : models\large_cnn.json + models\large_cnn.weights.h5


### Loading

In [ ]:
# Chargement
print("\n chargement du Large CNN")
loaded_large = load_keras_model("large_cnn")

# Recompilation
loaded_large.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Vérification
print("\nVérification après chargement :")
print_model_error_rate(loaded_large, X_test, y_test)


=== Test de chargement du Large CNN ===
Modèle chargé : models\large_cnn.json + models\large_cnn.weights.h5

Vérification après chargement :
Model score      : 99.09%
Model error rate : 0.91%
